In [30]:
import pandas as pd
import psycopg2

conn = psycopg2.connect(
    dbname="ProjetoMC536",
    user="postgres",
    password="GSW30_curry",
    host="localhost",
    port="5432"
)

cursor = conn.cursor()

In [8]:
categorias_conservacao = {
    "ESEC": "Estação Ecológica",
    "PARNA": "Parque Nacional",
    "REBIO": "Reserva Biológica",
    "RVS": "Refúgio de Vida Silvestre",
    "APA": "Área de Proteção Ambiental",
    "ARIE": "Área de Relevante Interesse Ecológico",
    "FLONA": "Floresta Nacional",
    "RDS": "Reserva de Desenvolvimento Sustentável",
    "RESEX": "Reserva Extrativista"
}

tipo_conservacao = {
    "Proteção Integral": ["ESEC", "PARNA", "REBIO", "RVS"],
    "Uso Sustentável": ["APA", "ARIE", "FLONA", "RDS", "RESEX"]
}

In [26]:
df = pd.read_csv('../datasets/CategoriaUnidadesConservacao.csv', sep=',')

df = df.rename(columns={
    'Categoria': 'sigla',
    'Quantidade': 'nome',
    'Área Oficial (ha)': 'area_total_amazonia_legal_ha',
    '% da Área em Relação à Área Total de Ucs': 'tipo'
})
df = df.drop(columns=['% da Área Oficial em relação a área AM Legal'])
df = df.drop(index=[0, 5, 6, 12, 13])
print(df.columns)

for index, row in df.iterrows():
    df.at[index, 'nome'] = categorias_conservacao[row['sigla']]
    df.at[index, 'area_total_amazonia_legal_ha'] = int(row['area_total_amazonia_legal_ha'].replace('.', ''))

    if row['sigla'] in tipo_conservacao["Proteção Integral"]:
        df.at[index, 'tipo'] = "Proteção Integral"
    elif row['sigla'] in tipo_conservacao["Uso Sustentável"]:
        df.at[index, 'tipo'] = "Uso Sustentável"
    # print(row)

data = list(df.itertuples(index=False, name=None))
print(data)

Index(['sigla', 'nome', 'area_total_amazonia_legal_ha', 'tipo'], dtype='object')
[('ESEC', 'Estação Ecológica', 7204872, 'Proteção Integral'), ('PARNA', 'Parque Nacional', 22925378, 'Proteção Integral'), ('REBIO', 'Reserva Biológica', 4069884, 'Proteção Integral'), ('RVS', 'Refúgio de Vida Silvestre', 15300, 'Proteção Integral'), ('APA', 'Área de Proteção Ambiental', 2605628, 'Uso Sustentável'), ('ARIE', 'Área de Relevante Interesse Ecológico', 20864, 'Uso Sustentável'), ('FLONA', 'Floresta Nacional', 17187140, 'Uso Sustentável'), ('RDS', 'Reserva de Desenvolvimento Sustentável', 64735, 'Uso Sustentável'), ('RESEX', 'Reserva Extrativista', 13077187, 'Uso Sustentável')]


/tmp/ipykernel_16047/2800659238.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Estação Ecológica' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[index, 'nome'] = categorias_conservacao[row['sigla']]


In [27]:
from psycopg2.extras import execute_values

# Agora, insere **em lote**:
query = """
    INSERT INTO public."CategoriaUnidadeConservacao" 
    (sigla, nome, area_total_amazonia_legal_ha, tipo)
    VALUES %s
"""
execute_values(cursor, query, data)

# Finaliza
conn.commit()
cursor.close()
conn.close()

In [28]:
data = [
    ["RPPN", "Reserva Particular do Patrimônio Natural", "Uso Sustentável"],
    ["MONA", "Monumento Natural", "Proteção Integral"],
    ]

In [31]:
query = """
    INSERT INTO public."CategoriaUnidadeConservacao" 
    (sigla, nome, tipo)
    VALUES %s
"""
execute_values(cursor, query, data)
# Finaliza
conn.commit()
cursor.close()
conn.close()

In [20]:
#Caso comando dê erro, desfaz as alterações
conn.rollback()